# AirSentinel Satellite Pull -- OPTIMIZED

**The one change that matters:** every pull/export step now checks Drive first and skips anything already there. Your old notebook re-submitted everything every run, which is how you ended up with ~2,800 duplicate queued tasks. Google's own docs confirm non-commercial accounts average only **2 concurrent batch tasks** -- no notebook can out-code that limit, so the fix is submitting fewer, not submitting faster.

Run top to bottom, in order. 20 cells total.

In [ ]:
!pip install earthengine-api geemap rasterio -q

## Connect to Earth Engine
Paste your real project ID below (not an API key).

In [ ]:
import ee
ee.Authenticate()
PROJECT_ID = 'YOUR_PROJECT_ID'
ee.Initialize(project=PROJECT_ID)
print('Connected:', PROJECT_ID)

## Zones and dates -- single source of truth
Every date you need, in one place. Verified CPCB coordinates.

In [ ]:
zones = {
    'Anand Vihar':   [77.315809, 28.647622],
    'Mundka':        [77.076574, 28.684678],
    'Wazirpur':      [77.165453, 28.699793],
    'Jahangirpuri':  [77.170633, 28.732820],
    'RK Puram':      [77.186937, 28.563262],
    'Rohini':        [77.119920, 28.732528],
    'Punjabi Bagh':  [77.131023, 28.674045],
    'Okhla':         [77.271255, 28.530785],
    'Bawana':        [77.051074, 28.776200],
    'Vivek Vihar':   [77.315260, 28.672342],
    'Narela':        [77.101981, 28.822836],
    'Ashok Vihar':   [77.181665, 28.695381],
    'Dwarka':        [77.071901, 28.571027],
}

from datetime import datetime, timedelta

def date_range(start, end, step_days):
    d = datetime.strptime(start, '%Y-%m-%d')
    e = datetime.strptime(end, '%Y-%m-%d')
    out = []
    while d <= e:
        out.append(d.strftime('%Y-%m-%d'))
        d += timedelta(days=step_days)
    return out

TARGET_DATES = sorted(set(
    date_range('2026-06-01', '2026-07-16', 5) +
    date_range('2025-10-01', '2025-11-26', 7) +
    date_range('2025-12-01', '2026-05-25', 5)
))

WINDOW_DAYS = 3
DRIVE_FOLDER = 'AirSentinel_Satellite_Images'

def date_window(date_str, window_days=WINDOW_DAYS):
    center = datetime.strptime(date_str, '%Y-%m-%d')
    return (center - timedelta(days=window_days)).strftime('%Y-%m-%d'), (center + timedelta(days=window_days)).strftime('%Y-%m-%d')

print(f'{len(zones)} zones x {len(TARGET_DATES)} dates = {len(zones) * len(TARGET_DATES)} max possible files per layer')

## Check what already exists -- THE core optimization
Mounts Drive and scans for every zone/date/layer file that already exists, so nothing gets re-pulled or re-exported. This single check is what prevents the duplicate-task pileup from happening again.

In [ ]:
import os, glob

try:
    from google.colab import drive
    drive.mount('/content/drive')
    drive_root = '/content/drive/MyDrive'
except ImportError:
    drive_root = 'PASTE_YOUR_LOCAL_GOOGLE_DRIVE_FOLDER_PATH_HERE'

target_dir = f'{drive_root}/{DRIVE_FOLDER}'
os.makedirs(target_dir, exist_ok=True)

existing_files = set(os.path.basename(f) for f in glob.glob(f'{target_dir}/**/*.tif', recursive=True))

def already_have(zone, date, layer):
    safe_name = zone.replace(' ', '_')
    return f'{safe_name}_{date}_{layer}.tif' in existing_files

missing_s2 = [(z, d) for z in zones for d in TARGET_DATES if not already_have(z, d, 'S2')]
missing_no2 = [(z, d) for z in zones for d in TARGET_DATES if not already_have(z, d, 'NO2')]

print(f'{len(existing_files)} files already exist -- will be skipped, not re-pulled')
print(f'Missing S2:  {len(missing_s2)} of {len(zones) * len(TARGET_DATES)}')
print(f'Missing NO2: {len(missing_no2)} of {len(zones) * len(TARGET_DATES)}')

## Check the Earth Engine task queue before submitting anything new
Cancel stale READY tasks first if this shows a large backlog -- don't add to a clogged queue.

In [ ]:
from collections import Counter
tasks = ee.batch.Task.list()
states = Counter(t.status()['state'] for t in tasks)
print(f'Total tasks in queue: {len(tasks)}')
print('By state:', dict(states))
if states.get('READY', 0) > 100:
    print('\nWARNING: large READY backlog. Consider cancelling stale tasks before submitting more:')
    print("for t in ee.batch.Task.list():\n    if t.status()['state'] == 'READY': t.cancel()")

## Pull S2 -- only what's missing

In [ ]:
zone_images_s2 = {}
for name, date in missing_s2:
    point = ee.Geometry.Point(zones[name])
    start, end = date_window(date)
    coll = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
            .filterBounds(point).filterDate(start, end).sort('CLOUDY_PIXEL_PERCENTAGE'))
    if coll.size().getInfo() == 0:
        continue
    zone_images_s2[(name, date)] = coll.first()
print(f'Pulled {len(zone_images_s2)} new S2 images (of {len(missing_s2)} attempted)')

## Pull NO2 -- only what's missing

In [ ]:
zone_images_no2 = {}
for name, date in missing_no2:
    point = ee.Geometry.Point(zones[name])
    region = point.buffer(5000).bounds()
    start, end = date_window(date)
    coll = (ee.ImageCollection('COPERNICUS/S5P/OFFL/L3_NO2')
            .select('tropospheric_NO2_column_number_density')
            .filterBounds(point).filterDate(start, end))
    if coll.size().getInfo() == 0:
        continue
    zone_images_no2[(name, date)] = coll.mean().clip(region)
print(f'Pulled {len(zone_images_no2)} new NO2 layers (of {len(missing_no2)} attempted)')

## Export -- only the new ones, nothing already in Drive

In [ ]:
for (name, date), image in zone_images_s2.items():
    region = ee.Geometry.Point(zones[name]).buffer(2000).bounds()
    safe_name = name.replace(' ', '_')
    ee.batch.Export.image.toDrive(
        image=image.select(['B2', 'B3', 'B4', 'B8A', 'B11', 'B12']),
        description=f'{safe_name}_{date}_S2', folder=DRIVE_FOLDER,
        fileNamePrefix=f'{safe_name}_{date}_S2', region=region, scale=10, maxPixels=1e9
    ).start()

for (name, date), image in zone_images_no2.items():
    region = ee.Geometry.Point(zones[name]).buffer(5000).bounds()
    safe_name = name.replace(' ', '_')
    ee.batch.Export.image.toDrive(
        image=image, description=f'{safe_name}_{date}_NO2', folder=DRIVE_FOLDER,
        fileNamePrefix=f'{safe_name}_{date}_NO2', region=region, scale=1000, maxPixels=1e9
    ).start()

print(f'Submitted {len(zone_images_s2)} S2 + {len(zone_images_no2)} NO2 tasks -- and NOTHING that already existed.')

## Sort into per-zone folders (run after exports land, safe to re-run anytime)

In [ ]:
import shutil
zone_names = [z.replace(' ', '_') for z in zones.keys()]
for zn in zone_names:
    os.makedirs(os.path.join(target_dir, zn), exist_ok=True)

all_tifs = glob.glob(f'{drive_root}/**/*.tif', recursive=True)
moved, skipped, failed = 0, 0, []
for f in all_tifs:
    try:
        fname = os.path.basename(f)
        zone = next((zn for zn in zone_names if fname.startswith(zn + '_')), None)
        if zone is None:
            continue
        dest = os.path.join(target_dir, zone, fname)
        if os.path.abspath(f) == os.path.abspath(dest):
            continue
        if os.path.exists(dest):
            skipped += 1
            continue
        shutil.move(f, dest)
        moved += 1
    except Exception as e:
        failed.append((f, str(e)))
print(f'Moved {moved}, skipped {skipped} duplicates, {len(failed)} failed.')

## Final verification -- real counts, real gaps

In [ ]:
final_s2 = glob.glob(f'{target_dir}/**/*_S2.tif', recursive=True)
final_no2 = glob.glob(f'{target_dir}/**/*_NO2.tif', recursive=True)
print(f'Real S2 files now on disk:  {len(final_s2)} of {len(zones) * len(TARGET_DATES)} possible')
print(f'Real NO2 files now on disk: {len(final_no2)} of {len(zones) * len(TARGET_DATES)} possible')